# Лабораторна робота №2
## Чисельне розв'язання рівняння Бюргерса

Параметри: $\beta=\gamma=1$, $T=50$, $\ell=100$, $h=1$, $\tau=1/3$.
Методи з умови: **ДС** [burgers.pdf], **неявна σ=1/2** (dis (2.3)) + Ньютон.
Додатково: **PINN** (TensorFlow + PhiFlow) за [physicalloss-code](https://physicsbaseddeeplearning.org/physicalloss-code.html).


## 1. Постановка


In [ ]:
# FDM: numpy, matplotlib, sympy
# PINN: tensorflow + phiflow (рекомендовано miniconda Python)
import sys, os
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..') if os.path.basename(os.getcwd())=='figures' else '.')
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from burgers_fdm import *


## 2. Аналітичний розв'язок


In [ ]:
x, N = build_grid()
t_sym, x_sym = sp.symbols('t x', real=True)
u01, u02, beta, gamma = 3, 1, 1, 1
c = (u01+u02)/2
f = (u01-u02)/(1+sp.exp(-(u01-u02)*x_sym/(2*u01)))
u = f.subs(x_sym, x_sym - c*t_sym)
res = sp.simplify(sp.diff(u,t_sym) + beta*u*sp.diff(u,x_sym) - gamma*sp.diff(u,x_sym,2))
print('Тестовий профіль f(x-ct); залишок PDE (символьно):', res)

plt.figure(figsize=(8,3))
plt.plot(x, u_exact(x,0), label='t=0')
plt.plot(x, u_exact(x,T_FINAL/2), label=f't={T_FINAL/2}')
plt.xlabel('x'); plt.ylabel('u'); plt.legend(); plt.grid(True, alpha=0.3)
plt.title('Точний розв\'язок'); plt.show()


## 3. Дискретизація


In [ ]:
print(f'N={N}, tau={TAU}, steps≈{int(T_FINAL/TAU)}')


## 4. Метод 1: ДС-алгоритм


In [ ]:
rec = max(1, int(T_FINAL/TAU)//30)
u_ds, t_ds, h_ds, n_ds, time_ds = run_solver(ds_burgers_step, TAU, T_FINAL, x, record_every=rec)
print(f'DS: steps={n_ds}, time={time_ds:.3f}s')


## 5. Метод 2: неявна схема


In [ ]:
u_im, t_im, h_im, n_im, time_im = run_solver(implicit_sigma05_newton_step, TAU, T_FINAL, x, record_every=rec)
print(f'Implicit sigma=0.5: steps={n_im}, time={time_im:.3f}s')


## 6. Експерименти FDM


In [ ]:
ue = u_exact(x, T_FINAL)
print('Linf DS', norm_linf(u_ds-ue), 'IM', norm_linf(u_im-ue))

# Згенерувати фігури для звіту (опційно):
# %run figures/_make_figures.py


## 7. PINN (TensorFlow + PhiFlow)

Структура як у [туторіалі](https://physicsbaseddeeplearning.org/physicalloss-code.html):
- `network(x,t)`, `f(u,x,t)` через `gradients`
- `loss = loss_u + ph_factor * loss_ph`
- `GradientDescentOptimizer(0.02)`, collocation


In [ ]:
# Потрібен Python з tensorflow та phiflow (див. burgers_pinn.py)
try:
    from burgers_pinn import train_pinn, ITERS_DEFAULT
    elapsed, hist, ug = train_pinn(iters=2000, log_every=500)
    u_pinn = np.asarray(ug)[0,:,:,0]
    print('PINN field', u_pinn.shape, 'final loss', hist[-1][1])
except Exception as e:
    print('PINN skip:', e)
    print('Запустіть: PINN_ITERS=3000 python figures/_make_figures.py')


## 8. Порівняння


In [ ]:
if 'u_pinn' in dir():
    e_p = u_pinn[:,-1] - ue
    print('PINN Linf', norm_linf(e_p), 'L2', norm_l2(e_p, H))


## 9. Висновки
- Два FDM-методи з умови реалізовано в `burgers_fdm.py`.
- PINN --- додатково в `burgers_pinn.py` (TF+PhiFlow).
- Звіт: `report.tex`, фігури: `figures/`.
